<a href="https://colab.research.google.com/github/KenPham0604/US_Accident_Severity_Prediction/blob/main/Spilt_Train_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Split Train/Test

In [ ]:
# 1 — Set up
from google.colab import drive
import pandas as pd
import numpy as np
import zipfile, shutil, os, gc

pd.set_option("display.max_columns", None)
SEED = 42
np.random.seed(SEED)
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/ADY201m/'

def optimize_ram(df, verbose=True):
    ram_before = df.memory_usage(deep=True).sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif col_type == "object":
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype("category")
    ram_after = df.memory_usage(deep=True).sum() / (1024**2)
    if verbose:
        print(f"RAM: {ram_before:.2f} MB -> {ram_after:.2f} MB")
    return df

def read_zip_by_chunking(file_name, drive_folder=DRIVE_FOLDER, chunksize=500_000):
    drive_zip = f'{drive_folder}{file_name}.zip'
    local_zip = f'/content/{file_name}.zip'
    shutil.copy(drive_zip, local_zip)
    with zipfile.ZipFile(local_zip, 'r') as z:
        csv_name = z.namelist()[0]
        chunks_list = []
        with z.open(csv_name) as f:
            for i, chunk in enumerate(pd.read_csv(f, chunksize=chunksize)):
                chunks_list.append(chunk)
                print(f"  Chunk {i+1}: {sum(len(c) for c in chunks_list):,} rows")
    df = pd.concat(chunks_list, ignore_index=True)
    del chunks_list
    gc.collect()
    os.remove(local_zip)
    return df

def save_csv_as_zip(df_to_save, file_name, drive_folder=DRIVE_FOLDER):
    local_csv = f'/content/{file_name}.csv'
    local_zip = f'/content/{file_name}.zip'
    drive_zip = f'{drive_folder}{file_name}.zip'
    df_to_save.to_csv(local_csv, index=False)
    with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(local_csv, arcname=f'{file_name}.csv')
    shutil.copy(local_zip, drive_zip)
    print(f"✅ Đã lưu: {drive_zip} ({df_to_save.shape[0]:,} rows)")
    os.remove(local_csv)
    os.remove(local_zip)
    gc.collect()
    return drive_zip

df = read_zip_by_chunking('US_Accidents_processed_full')
print(f"\nRestore size: {df.shape}")

df = optimize_ram(df)
gc.collect()

print(f"\n✅ Restored 'df' — Ready to proceed with Part 2 (Train/Test split).")
print(f"List of columns:\n{list(df.columns)}")

In [ ]:
# Random split

In [ ]:
# 2 — Random stratification based on severity and season
from sklearn.model_selection import train_test_split

df["strat_key"] = df["Severity"].astype(str) + "_" + df["Season"].astype(str)

df_train_random, df_test_random = train_test_split(
    df,
    test_size=0.2,
    stratify=df["strat_key"],
    random_state=SEED
)

df_train_random = df_train_random.drop(columns=["strat_key"])
df_test_random = df_test_random.drop(columns=["strat_key"])

print(f"Train (random): {df_train_random.shape}")
print(f"Test (random):  {df_test_random.shape}")
gc.collect()

In [ ]:
# 3 — Fit encoder on Train (random):
# Frequency Encoding for Street, Zipcode
# Label Encoding for City, County, State, Timezone, Airport_Code,
#                     Wind_Direction, Weather_Condition, Season

import joblib

col_frequency = ["Street", "Zipcode"]
col_label = ["City", "County", "State", "Timezone", "Airport_Code",
             "Wind_Direction", "Weather_Condition", "Season"]

freq_maps_random = {}
for col in col_frequency:
    freq_map = df_train_random[col].value_counts().to_dict()
    freq_maps_random[col] = freq_map
    df_train_random[col + "_freq"] = df_train_random[col].map(freq_map).astype("float32")
    print(f"  {col}: were caculated Frequency Encoding on Train (random)")

from sklearn.preprocessing import LabelEncoder

label_encoders_random = {}
for col in col_label:
    le = LabelEncoder()
    df_train_random[col] = df_train_random[col].astype(str)
    df_train_random[col + "_encoded"] = le.fit_transform(df_train_random[col])
    label_encoders_random[col] = le
    print(f"  {col}: fit Label Encoding on Train (random), {len(le.classes_)} class")

df_train_random = df_train_random.drop(columns=col_frequency + col_label)

gc.collect()
print(f"\nShape of Train (random) after encode: {df_train_random.shape}")

In [ ]:
# 4 — Save encoder (random) + save file Train (random) encoded

joblib.dump(freq_maps_random, '/content/freq_maps_random.pkl')
joblib.dump(label_encoders_random, '/content/label_encoders_random.pkl')

shutil.copy('/content/freq_maps_random.pkl', f'{DRIVE_FOLDER}freq_maps_random.pkl')
shutil.copy('/content/label_encoders_random.pkl', f'{DRIVE_FOLDER}label_encoders_random.pkl')
os.remove('/content/freq_maps_random.pkl')
os.remove('/content/label_encoders_random.pkl')
print("✅ Save encoder (random) into Drive")

save_csv_as_zip(df_train_random, 'US_Accidents_train_random')
del df_train_random
gc.collect()

In [ ]:
# 5 — Apply the (random) encoder fitted on the Train (random) set to the Test (random) set.
# Frequency Encoding: map using the frequency table calculated from the training set;
# unseen values ​​(present only in the test set) -> assign frequency = 0
for col in col_frequency:
    df_test_random[col + "_freq"] = df_test_random[col].map(freq_maps_random[col]).fillna(0).astype("float32")

# Label Encoding: reuse the fitted encoder; assign "Unknown" to unseen values
# (need to add "Unknown" to classes_ if not already present)
for col in col_label:
    le = label_encoders_random[col]
    df_test_random[col] = df_test_random[col].astype(str)
    known_classes = set(le.classes_)

    if "Unknown" not in known_classes:
        le.classes_ = np.append(le.classes_, "Unknown")

    test_values = df_test_random[col].apply(lambda x: x if x in known_classes else "Unknown")
    df_test_random[col + "_encoded"] = le.transform(test_values)

df_test_random = df_test_random.drop(columns=col_frequency + col_label)

print(f"Shape of Test (random) sau encode: {df_test_random.shape}")
gc.collect()

save_csv_as_zip(df_test_random, 'US_Accidents_test_random')
del df_test_random
gc.collect()

print("\n✅ Random split complete — saved 2 file Train/Test.")